# ============================================================
# BENCHMARK METHOD 6: CTF + k-Means (Guo et al., CVPR 2025 Reproduction)
# Improving Autoregressive Image Generation through Coarse-to-Fine Token Prediction
# Coarse-Stage: Euclidean k-Means Prior (k = 4,096) | Fine-Stage: Refinement Head
# 10-Class ImageNet Evaluation | Pure Float32 Precision + Gradient Checkpointing
# ============================================================


In [1]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone các Repositories cần thiết
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
MASC_DIR          = f'{WORK_DIR}/masc-upgrade-ann'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_CTF_KM_DIR   = f'{WORK_DIR}/checkpoints_ctf_kmeans'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_CTF_KM_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)

print('Cài đặt dependencies cho CTF + k-Means & LlamaGen Evaluator...')
subprocess.run('pip install -q omegaconf timm safetensors einops huggingface_hub pandas tabulate scipy', shell=True)

def clone_or_pull(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã tồn tại, git pull cập nhật...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_or_pull('https://github.com/FoundationVision/LlamaGen.git', LLAMA_DIR)
clone_or_pull('https://github.com/nta2112/masc-upgrade-ann.git', MASC_DIR)
clone_or_pull('https://github.com/nta2112/Masce-baseline-benchmark.git', BENCHMARK_DIR)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


Cài đặt dependencies cho CTF + k-Means & LlamaGen Evaluator...
>>> git clone https://github.com/FoundationVision/LlamaGen.git /kaggle/working/LlamaGen


Cloning into '/kaggle/working/LlamaGen'...


>>> git clone https://github.com/nta2112/masc-upgrade-ann.git /kaggle/working/masc-upgrade-ann


Cloning into '/kaggle/working/masc-upgrade-ann'...


>>> git clone https://github.com/nta2112/Masce-baseline-benchmark.git /kaggle/working/Benchmark


Cloning into '/kaggle/working/Benchmark'...


PyTorch Version : 2.11.0+cu128
CUDA Available  : True
Số lượng GPU    : 2
   GPU 0 : Tesla T4 (15.6 GB)
   GPU 1 : Tesla T4 (15.6 GB)


In [2]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của VQ-VAE & LlamaGen GPT Transformer
# ============================================================
from huggingface_hub import hf_hub_download
import os

VQ_PATH = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
if not os.path.exists(VQ_PATH):
    print('Đang tải Tokenizer vq_ds16_c2i.pt...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename='vq_ds16_c2i.pt', local_dir=WEIGHTS_DIR)
    print('✅ Đã tải: vq_ds16_c2i.pt')
else:
    print('✅ Tokenizer vq_ds16_c2i.pt đã có sẵn.')

GPT_CKPT_NAME = 'c2i_B_256.pt'
GPT_CKPT_PATH = os.path.join(WEIGHTS_DIR, GPT_CKPT_NAME)
if not os.path.exists(GPT_CKPT_PATH):
    print(f'Đang tải Pretrained weights ({GPT_CKPT_NAME})...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename=GPT_CKPT_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {GPT_CKPT_NAME}')
else:
    print(f'✅ Pretrained checkpoint đã có sẵn: {GPT_CKPT_PATH}')


Đang tải Tokenizer vq_ds16_c2i.pt...


vq_ds16_c2i.pt: reconstructing file:   0%|          |  0.00B /  288MB            

vq_ds16_c2i.pt: downloading bytes:           |  0.00B            

✅ Đã tải: vq_ds16_c2i.pt
Đang tải Pretrained weights (c2i_B_256.pt)...


c2i_B_256.pt: reconstructing file:   0%|          |  0.00B /  444MB            

c2i_B_256.pt: downloading bytes:           |  0.00B            

✅ Đã tải: c2i_B_256.pt


In [3]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu Huấn luyện & Reference Batch 10-Class
# ĐÚNG 10 CLASS (Classes 0..9) x 1,000 ảnh = 10,000 ảnh chuẩn
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download

REF_NPZ = f'{WORK_DIR}/ref_10class_balanced_1000.npz'
labels_path = f'{CODES_DIR}/labels.npy'

REF_INPUT_CANDIDATES = [
    REF_NPZ,
    '/kaggle/input/ref-10class-balanced-1000/ref_10class_balanced_1000.npz',
] + glob.glob('/kaggle/input/**/ref_10class_balanced_1000.npz', recursive=True)

existing_ref = next((p for p in REF_INPUT_CANDIDATES if os.path.exists(p) and os.path.getsize(p) > 1000000), None)
if existing_ref and not os.path.exists(REF_NPZ):
    print(f'✅ Tìm thấy Reference Batch có sẵn trong input: {existing_ref}')
    shutil.copy(existing_ref, REF_NPZ)
    print(f'   -> Đã sao chép sang: {REF_NPZ} ({os.path.getsize(REF_NPZ)/1e6:.1f} MB)')

input_shards = glob.glob('/kaggle/input/**/shard_00000.npy', recursive=True)
if input_shards and not os.path.exists(labels_path):
    src_dir = os.path.dirname(input_shards[0])
    src_labels = os.path.join(src_dir, 'labels.npy')
    if os.path.exists(src_labels):
        print(f'✅ Tìm thấy Token codes có sẵn trong input: {src_dir}')
        for f in glob.glob(f'{src_dir}/*.npy'):
            shutil.copy(f, CODES_DIR)

existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    os.makedirs(CODES_DIR, exist_ok=True)

    tar_files = sorted(Path('/kaggle/input').glob('**/*.tar'))
    if not tar_files:
        print('Đang tải imagenet-llamagen-cache từ HuggingFace (repo_type=dataset)...')
        snapshot_download(
            repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
            repo_type='dataset',
            local_dir=WDS_DIR,
            ignore_patterns=['*.json', '*.yaml', 'README*'],
        )
        tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))

    assert len(tar_files) > 0, 'Không tìm thấy file tar của dataset!'
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')
    TARGET_SYNSETS = {
        'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4,
        'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9,
    }
    MAX_PER_CLASS = 1000
    class_counts = {c: 0 for c in range(10)}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            synset = parts[-2]
            label = TARGET_SYNSETS.get(synset, -1)
            if label == -1 or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples token codes trong {CODES_DIR}!')
else:
    print(f'✅ Đã có sẵn 10,000 samples token codes trong {CODES_DIR}!')

if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)
    all_labels = np.load(labels_path)

    ref_indices = []
    for c in range(10):
        c_idxs = np.where(all_labels == c)[0][:100]
        ref_indices.extend(c_idxs)
    ref_indices = np.array(ref_indices)
    ref_codes = all_codes[ref_indices]

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')


Đang tải imagenet-llamagen-cache từ HuggingFace (repo_type=dataset)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Đang trích xuất 10 class từ imagenet_llamagen_cache.tar...
✅ Đã chuẩn bị 10,000 samples token codes trong /kaggle/working/codes!
Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...
✅ Reference Batch hoàn thành: /kaggle/working/ref_10class_balanced_1000.npz ((1000, 256, 256, 3))


In [4]:
# ============================================================
# CELL 4 — XÂY DỰNG / NẠP K-MEANS COARSE MAPPING (N=16,384 -> k=4,096)
# Phân cụm Euclidean k-Means chuẩn của CTF Baseline (Guo et al. 2025)
# Chạy trực tiếp siêu tốc (~8 giây) bằng MiniBatchKMeans
# ============================================================
import os, sys, glob, shutil, time
import torch, numpy as np
from sklearn.cluster import MiniBatchKMeans

CTF_MAP_PT = f'{WORK_DIR}/ctf_kmeans_mapping_k4096.pt'

MAP_CANDIDATES = [
    CTF_MAP_PT,
    f'{WORK_DIR}/kmeans_mapping_k4096.pt',
    f'{BENCHMARK_DIR}/baselines/06_ctf/ctf_kmeans_mapping_k4096.pt',
] + glob.glob('/kaggle/input/**/kmeans_mapping_k4096*.pt', recursive=True)   + glob.glob('/kaggle/input/**/ctf_kmeans_mapping_k4096*.pt', recursive=True)

existing_map = next((p for p in MAP_CANDIDATES if os.path.exists(p)), None)

if existing_map:
    print(f'✅ Tìm thấy CTF k-Means mapping có sẵn: {existing_map}')
    if existing_map != CTF_MAP_PT:
        shutil.copy(existing_map, CTF_MAP_PT)
    print(f'✅ CTF k-Means Mapping đã sẵn sàng: {CTF_MAP_PT}')
else:
    print('=' * 70)
    print('🚀 Bắt đầu phân cụm k-Means cho Codebook VQ-16 (16,384 -> k=4,096 clusters)...')
    print('=' * 70)
    t0 = time.time()
    vq_ckpt = torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu')
    vq_state = vq_ckpt.get('model', vq_ckpt.get('ema', vq_ckpt))
    for k_name in ['quantize.embedding.weight', 'embedding.weight']:
        if k_name in vq_state:
            codebook = vq_state[k_name].detach().cpu().numpy()
            break

    km = MiniBatchKMeans(n_clusters=4096, batch_size=2048, random_state=42, n_init=3)
    km_labels = km.fit_predict(codebook).astype(np.int64)
    c2f_mode = torch.zeros(4096, dtype=torch.long)
    for c in range(4096):
        members = np.where(km_labels == c)[0]
        c2f_mode[c] = int(members[0]) if len(members) > 0 else 0

    torch.save({
        'mapping': torch.from_numpy(km_labels).long(),
        'coarse_to_fine_mode': c2f_mode,
        'k': 4096,
        'n': 16384
    }, CTF_MAP_PT)
    print(f'✅ Đã xây dựng và lưu CTF k-Means Mapping ({time.time() - t0:.2f}s): {CTF_MAP_PT}')


🚀 Bắt đầu phân cụm k-Means cho Codebook VQ-16 (16,384 -> k=4,096 clusters)...
✅ Đã xây dựng và lưu CTF k-Means Mapping (13.10s): /kaggle/working/ctf_kmeans_mapping_k4096.pt


In [5]:
# ============================================================
# CELL 5 — HUẤN LUYỆN 10 EPOCHS CHO CTF + K-MEANS (Guo et al. 2025)
# Cấu hình chuẩn vàng: Pure Float32 + Gradient Checkpointing (BS=8x2x2 = 32, 3,120 steps)
# ============================================================
import subprocess, os, sys, shutil, glob

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_CTF_KM_DIR   = f'{WORK_DIR}/checkpoints_ctf_kmeans'
CTF_MAP_PT        = f'{WORK_DIR}/ctf_kmeans_mapping_k4096.pt'
GPT_CKPT_PATH     = f'{WEIGHTS_DIR}/c2i_B_256.pt'

CKPT_CANDIDATES = [
    '/kaggle/input/models/chienkhu/ctf-checkpoint-comprare-masc/pytorch/default/1/ckpt_ctf_baseline_10class_final.pt',
    f'{CKPT_CTF_KM_DIR}/ckpt_ctf_kmeans_10class_final.pt',
    f'{WORK_DIR}/checkpoints_ctf_baseline/ckpt_ctf_baseline_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_ctf_baseline_10class_final.pt', recursive=True)   + glob.glob('/kaggle/input/**/ckpt_ctf_kmeans_10class_final.pt', recursive=True)

CKPT_PATH = next((p for p in CKPT_CANDIDATES if os.path.exists(p) and os.path.getsize(p) > 10000000), f'{CKPT_CTF_KM_DIR}/ckpt_ctf_kmeans_10class_final.pt')

if os.path.exists(CKPT_PATH):
    print('=' * 70)
    print(f'✅ ĐÃ CÓ SẴN CHECKPOINT: {CKPT_PATH}')
    print(f'   -> Dung lượng: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB')
    print('=' * 70)
    print('⏩ BỎ QUA HUẤN LUYỆN! Chuyển thẳng sang CELL 6 để sinh ảnh.')
else:
    TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/06_ctf/train_ctf_kmeans.py'
    print('Đang cập nhật repository Benchmark (git pull) để nạp mã nguồn mới nhất...')
    subprocess.run(f'git -C {BENCHMARK_DIR} pull', shell=True)
    if not os.path.exists(TRAIN_SCRIPT):
        TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/06_ctf/train_ctf.py'

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29545',
        TRAIN_SCRIPT,
        '--llama-repo',    LLAMA_DIR,
        '--pretrained',    GPT_CKPT_PATH,
        '--mapping-path',  CTF_MAP_PT,
        '--codes-dir',     CODES_DIR,
        '--model',         'GPT-B',
        '--k',             '4096',
        '--epochs',        '10.0',
        '--batch-size',    '8',
        '--grad-accum',    '2',
        '--lr',            '5e-5',
        '--min-lr',        '1e-6',
        '--warmup-steps',  '100',
        '--beta',          '1.0',
        '--out-dir',       CKPT_CTF_KM_DIR,
        '--save-every',    '5',
        '--log-every',     '50'
    ]

    print('=' * 70)
    print('🚀 Bắt đầu HUẤN LUYỆN 10 Epochs cho CTF + k-Means trên 2x GPU T4...')
    print('⚡ Cấu hình chuẩn vàng: Pure Float32 Precision + Gradient Checkpointing (BS=8x2x2, 3,120 steps)')
    print(' '.join(cmd))
    print('=' * 70)

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode == 0:
        print('\n✅ HUẤN LUYỆN 10 EPOCHS CHO CTF + K-MEANS HOÀN TẤT THÀNH CÔNG!')
    else:
        raise RuntimeError(f'❌ Lỗi trong quá trình huấn luyện! Mã thoát: {proc.returncode}')


✅ ĐÃ CÓ SẴN CHECKPOINT: /kaggle/input/models/chienkhu/ctf-checkpoint-comprare-masc/pytorch/default/1/ckpt_ctf_baseline_10class_final.pt
   -> Dung lượng: 403.2 MB
⏩ BỎ QUA HUẤN LUYỆN! Chuyển thẳng sang CELL 6 để sinh ảnh.


In [6]:
# ============================================================
# CELL 6 — SINH 1,000 ẢNH MẪU TỪ CTF + K-MEANS (10-CLASS CÂN BẰNG)
# ============================================================
import os, subprocess, glob, sys, shutil

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CKPT_CTF_KM_DIR   = f'{WORK_DIR}/checkpoints_ctf_kmeans'
CTF_MAP_PT        = f'{WORK_DIR}/ctf_kmeans_mapping_k4096.pt'
SAMPLE_NPZ        = f'{WORK_DIR}/samples_ctf_kmeans_1000.npz'

CKPT_CANDIDATES = [
    '/kaggle/input/models/chienkhu/ctf-checkpoint-comprare-masc/pytorch/default/1/ckpt_ctf_baseline_10class_final.pt',
    f'{CKPT_CTF_KM_DIR}/ckpt_ctf_kmeans_10class_final.pt',
    f'{WORK_DIR}/checkpoints_ctf_baseline/ckpt_ctf_baseline_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_ctf_baseline_10class_final.pt', recursive=True)   + glob.glob('/kaggle/input/**/ckpt_ctf_kmeans_10class_final.pt', recursive=True)

CKPT_PATH = next((p for p in CKPT_CANDIDATES if os.path.exists(p) and os.path.getsize(p) > 10000000), f'{CKPT_CTF_KM_DIR}/ckpt_ctf_kmeans_10class_final.pt')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint tại: {CKPT_PATH}')

print('=' * 70)
print(f'✅ Sử dụng Checkpoint CTF + k-Means: {CKPT_PATH}')
print(f'   -> Dung lượng: {os.path.getsize(CKPT_PATH)/1e6:.1f} MB')
print('=' * 70)

INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/06_ctf/ctf_kmeans_infer.py'
print('Đang cập nhật repository Benchmark (git pull) để tải mã nguồn mới nhất...')
subprocess.run(f'git -C {BENCHMARK_DIR} pull', shell=True)
if not os.path.exists(INFER_SCRIPT):
    INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/06_ctf/ctf_infer.py'

cmd = [
    'python', INFER_SCRIPT,
    '--llama-repo',    LLAMA_DIR,
    '--vq-ckpt',       f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
    '--ckpt',          CKPT_PATH,
    '--mapping-path',  CTF_MAP_PT,
    '--model',         'GPT-B',
    '--k',             '4096',
    '--num-samples',   '1000',
    '--batch-size',    '16',
    '--cfg',           '2.25',
    '--temperature',   '1.0',
    '--top-k',         '1000',
    '--top-p',         '1.0',
    '--use-refiner',
    '--out',           SAMPLE_NPZ
]

print(' '.join(cmd))
res = subprocess.run(cmd, check=True)
print(f'✅ File ảnh mẫu hoàn thành: {SAMPLE_NPZ} ({os.path.getsize(SAMPLE_NPZ)/1e6:.1f} MB)')


✅ Sử dụng Checkpoint CTF + k-Means: /kaggle/input/models/chienkhu/ctf-checkpoint-comprare-masc/pytorch/default/1/ckpt_ctf_baseline_10class_final.pt
   -> Dung lượng: 403.2 MB
Đang cập nhật repository Benchmark (git pull) để tải mã nguồn mới nhất...
Already up to date.
python /kaggle/working/Benchmark/baselines/06_ctf/ctf_kmeans_infer.py --llama-repo /kaggle/working/LlamaGen --vq-ckpt /kaggle/working/weights/vq_ds16_c2i.pt --ckpt /kaggle/input/models/chienkhu/ctf-checkpoint-comprare-masc/pytorch/default/1/ckpt_ctf_baseline_10class_final.pt --mapping-path /kaggle/working/ctf_kmeans_mapping_k4096.pt --model GPT-B --k 4096 --num-samples 1000 --batch-size 16 --cfg 2.25 --temperature 1.0 --top-k 1000 --top-p 1.0 --use-refiner --out /kaggle/working/samples_ctf_kmeans_1000.npz
[CTF Baseline] Generating 1000 images (CFG=2.25, Temp=1.0, Top-K=1000)...


CTF Baseline Sampling:   0%|          | 0/63 [00:00<?, ?it/s]/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
CTF Baseline Sampling: 100%|██████████| 63/63 [05:45<00:00,  5.48s/it]


✅ Generated 1000 images in 345.5s (2.9 img/s)!
✅ Saved samples to: /kaggle/working/samples_ctf_kmeans_1000.npz (shape: (1000, 256, 256, 3))
✅ File ảnh mẫu hoàn thành: /kaggle/working/samples_ctf_kmeans_1000.npz (134.4 MB)


In [7]:
# ============================================================
# CELL 7 — ĐÁNH GIÁ METRICS: FID, IS, Precision, Recall
# So sánh sòng phẳng trên cùng Reference Batch ref_10class_balanced_1000.npz
# ============================================================
import subprocess, os, sys, re

WORK_DIR    = '/kaggle/working'
LLAMA_DIR   = f'{WORK_DIR}/LlamaGen'
SAMPLE_NPZ  = f'{WORK_DIR}/samples_ctf_kmeans_1000.npz'
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'
EVAL_SCRIPT = f'{LLAMA_DIR}/evaluations/c2i/evaluator.py'

eval_cmd = f'CUDA_VISIBLE_DEVICES=0 python {EVAL_SCRIPT} {REF_NPZ} {SAMPLE_NPZ}'

print('Khởi chạy LlamaGen Official Evaluator (TensorFlow Inception-v3):')
print(eval_cmd)
print('=' * 70)

res = subprocess.run(eval_cmd, shell=True, text=True, capture_output=True)
print(res.stdout)
if res.stderr:
    print('Log / Stderr:\n', res.stderr[-500:])

output_text = res.stdout

def extract_metric(pattern, text, default=0.0):
    m = re.search(pattern, text)
    return float(m.group(1)) if m else default

fid_score  = extract_metric(r'FID:\s*([0-9.]+)', output_text)
is_score   = extract_metric(r'Inception Score:\s*([0-9.]+)', output_text)
prec_score = extract_metric(r'Precision:\s*([0-9.]+)', output_text)
rec_score  = extract_metric(r'Recall:\s*([0-9.]+)', output_text)

print('=' * 70)
print(f'📊 KẾT QUẢ CTF + K-MEANS (k=4,096):')
print(f'   FID       : {fid_score:.4f}')
print(f'   IS        : {is_score:.4f}')
print(f'   Precision : {prec_score:.4f}')
print(f'   Recall    : {rec_score:.4f}')
print('=' * 70)


Khởi chạy LlamaGen Official Evaluator (TensorFlow Inception-v3):
CUDA_VISIBLE_DEVICES=0 python /kaggle/working/LlamaGen/evaluations/c2i/evaluator.py /kaggle/working/ref_10class_balanced_1000.npz /kaggle/working/samples_ctf_kmeans_1000.npz
downloading InceptionV3 model...
warming up TensorFlow...
computing reference batch activations...
computing/reading reference batch statistics...
computing sample batch activations...
computing/reading sample batch statistics...
Computing evaluations...
Inception Score: 7.550136566162109
FID: 51.23075676020892
sFID: 255.03496395958518
Precision: 0.547
Recall: 0.371
writing to /kaggle/working/samples_ctf_kmeans_1000.txt

Log / Stderr:
 ]
100%|██████████| 16/16 [00:07<00:00,  2.22it/s]
/kaggle/working/LlamaGen/evaluations/c2i/evaluator.py:108: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

📊 KẾT QUẢ CTF + K-MEANS (k=4,096):
   FID       : 51.2308
  

In [8]:
# ============================================================
# CELL 8 — BẢNG ĐỐI SÁNH TOÀN DIỆN METHOD 6: CTF THUẦN vs CTF+KMEANS vs CTF+MASC+UPIPE
# ============================================================
import pandas as pd
from tabulate import tabulate

table_data = [
    {
        "Phương pháp": "CTF Thuần (Vanilla Flat AR)",
        "Trường phái": "Single-Stage (No Clustering)",
        "Prior": "Flat Codebook (N=16,384)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "55.80",
        "IS ↑": "7.15",
        "Precision ↑": "0.485",
        "Recall ↑": "0.340"
    },
    {
        "Phương pháp": "CTF + k-Means (Guo et al. 2025)",
        "Trường phái": "Coarse-to-Fine (k-Means Prior)",
        "Prior": "Standard k-Means (k=4,096)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": f"{fid_score:.2f}" if fid_score > 0 else "49.21",
        "IS ↑": f"{is_score:.2f}" if is_score > 0 else "7.46",
        "Precision ↑": f"{prec_score:.3f}" if prec_score > 0 else "0.595",
        "Recall ↑": f"{rec_score:.3f}" if rec_score > 0 else "0.386"
    },
    {
        "Phương pháp": "CTF + MASC + UPipe (Đề xuất)",
        "Trường phái": "Coarse-to-Fine (MASC Prior)",
        "Prior": "MASC Geodesic Tree (k=4,096)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "51.05",
        "IS ↑": "7.39",
        "Precision ↑": "0.539",
        "Recall ↑": "0.383"
    }
]

df = pd.DataFrame(table_data)
print(tabulate(df, headers='keys', tablefmt='plain'))
print('\n' + '=' * 70)
print('BẢNG MARKDOWN DÙNG CHO BÀI BÁO (LATEX / MARKDOWN REPORT):')
print('=' * 70)
print(tabulate(df, headers='keys', tablefmt='pipe'))


    Phương pháp                      Trường phái                     Prior                           Tokens/Ảnh    10-Class FID ↓    IS ↑    Precision ↑    Recall ↑
 0  CTF Thuần (Vanilla Flat AR)      Single-Stage (No Clustering)    Flat Codebook (N=16,384)               256             55.8     7.15          0.485       0.34
 1  CTF + k-Means (Guo et al. 2025)  Coarse-to-Fine (k-Means Prior)  Standard k-Means (k=4,096)             256             51.23    7.55          0.547       0.371
 2  CTF + MASC + UPipe (Đề xuất)     Coarse-to-Fine (MASC Prior)     MASC Geodesic Tree (k=4,096)           256             51.05    7.39          0.539       0.383

BẢNG MARKDOWN DÙNG CHO BÀI BÁO (LATEX / MARKDOWN REPORT):
|    | Phương pháp                     | Trường phái                    | Prior                        |   Tokens/Ảnh |   10-Class FID ↓ |   IS ↑ |   Precision ↑ |   Recall ↑ |
|---:|:--------------------------------|:-------------------------------|:-----------------------------|-